# 7. Multi-instrument joint analysis

## Aim
The aim of this tutorial is to do a joint spectral analysis using H.E.S.S. and Fermi-LAT data of AGN PKS 2155-305.

In [ ]:
from gammapy.data import DataStore, EventList
from gammapy.irf import PSFMap, EDispKernelMap
from gammapy.maps import MapAxis, RegionGeom, WcsGeom, Map
from gammapy.makers import SpectrumDatasetMaker, SafeMaskMaker, ReflectedRegionsBackgroundMaker
from gammapy.datasets import SpectrumDataset, Datasets, FluxPointsDataset, MapDataset
from gammapy.catalog import CATALOG_REGISTRY, SourceCatalog4FGL
from gammapy.modeling.models import (
    Models,
    PowerLawSpectralModel,
    SkyModel,
    EBLAbsorptionNormSpectralModel, 
    ExpCutoffPowerLawSpectralModel,
    create_fermi_isotropic_diffuse_model,
    TemplateSpatialModel,
    PowerLawNormSpectralModel
)
from gammapy.modeling import Fit
from gammapy.estimators import FluxPointsEstimator

from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.time import Time
from regions import CircleSkyRegion

import matplotlib.pyplot as plt

## H.E.S.S.

Select H.E.S.S. observations for PKS 2155-304 that you can find in the Gammapy datasets in `$GAMMAPY_DATA`

In [ ]:
data_store = DataStore.from_dir("$GAMMAPY_DATA/hess-dl3-dr1/")

In [ ]:
target_pos = XXX

In [ ]:
obs_time = Time(["2008-08-26", "2008-09-06"], scale="tt")

In [ ]:
obs_table = data_store.obs_table
obs_table = obs_table.select_sky_circle
obs_table = obs_table.select_time_range

Display the content of the `obs_table`

In [ ]:
XXX

In [ ]:
len(obs_table)

Get the observations

In [ ]:
observations = XXX

In [ ]:
print(observations)

### Data reduction to DL4

In [ ]:
on_region = CircleSkyRegion(target_pos, 0.1 * u.deg)

In [ ]:
energy_axis = MapAxis.from_energy_bounds(
    0.1, 40, nbin=10, per_decade=True, unit="TeV", name="energy"
)
energy_axis_true = MapAxis.from_energy_bounds(
    0.05, 100, nbin=20, per_decade=True, unit="TeV", name="energy_true"
)

geom = RegionGeom.create(region=on_region, axes=[energy_axis])
dataset_empty = SpectrumDataset.create(geom=geom, energy_axis_true=energy_axis_true)

dataset_maker = SpectrumDatasetMaker(
    containment_correction=True, selection=["counts", "exposure", "edisp"]
)
bkg_maker = ReflectedRegionsBackgroundMaker()
safe_mask_masker = SafeMaskMaker(methods=["aeff-max"], aeff_percent=10)

In [ ]:
obs_ids = observations.ids
datasets = Datasets()

for obs_id, observation in zip(obs_ids, observations):
    dataset = dataset_maker.run(dataset_empty.copy(name=str(obs_id)), observation)
    dataset_on_off = bkg_maker.run(dataset, observation)
    dataset_on_off = safe_mask_masker.run(dataset_on_off, observation)
    datasets.append(dataset_on_off)

print(datasets)

Stack the dataset using `stack_reduce`

In [ ]:
hess_dataset = 

## Fermi-LAT

This is a simplified version. For more details on real Fermi-LAT analysis with Gammapy see https://docs.gammapy.org/2.0/tutorials/data/fermi_lat.html

Besides the files already provided in the repository, you need to download Galactic interstellar emission model `gll_iem_v07.fits` from https://fermi.gsfc.nasa.gov/ssc/data/access/lat/BackgroundModels.html as they are large files.

Read the events using `EventList.read`. Fermi-LAT events are stored in the file "ft1_00.fits"

In [ ]:
events = 

In [ ]:
print(events)

In [ ]:
events.table

In [ ]:
events.peek()

create an energy axis from 0.2 to 300 GeV with 8 bins

In [ ]:
energy_axis = 
print(energy_axis)

Create a `WcsGeom` with a width of 10 deg and 0.1 deg per bin centered on the source position

In [ ]:
geom = WcsGeom.create(
    skydir=,
    binsz=,
    width=,
    axes=,
)
print(geom)

Create a counts map using this geometry

In [ ]:
counts = 
print(counts)

Fill the counts map with the event using `Map.fill_events`.

In [ ]:
counts.fill_events(events)

Have a look at the resulting count maps:

In [ ]:
counts.plot_grid();

Read the exposure using a `Map.read`. The exposure is stored in 'bexpmap_roi_00.fits'.

In [ ]:
exposure = 

In [ ]:
print(exposure)

In [ ]:
energy_true = exposure.geom.axes["energy_true"]

In [ ]:
irf_geom = WcsGeom(wcs=counts.geom.wcs, npix=counts.geom.npix, axes=[energy_true])

In [ ]:
exposure = exposure.interp_to_geom(irf_geom)
print(exposure)

In [ ]:
exposure.sum_over_axes().plot(add_cbar=True)

Read the psf using `PSFMap.read`. The psf is stored in 'psf_00.fits'.

In [ ]:
psf = 
print(psf)

In [ ]:
psf_kernel = psf.get_psf_kernel(
    position=irf_geom.center_skydir,
    geom=irf_geom,
    max_radius="3 deg"
)
psf_kernel.psf_kernel_map.sum_over_axes().plot(stretch='log', add_cbar=True)

Create a diagonal energy dispersion matrix using `EDispKernelMap.from_diagonal_response` and the energy axes of the counts map

In [ ]:
edisp = EDispKernelMap.from_diagonal_response(
    energy_axis=,
    energy_axis_true=,
)
edisp.get_edisp_kernel().plot_matrix()

Create the fermi dataset by putting the counts and IRFs that we have created above into a `MapDataset`. Call it `"fermi-lat"`.

In [ ]:
fermi_dataset = MapDataset(
    counts=,
    exposure=,
    edisp=,
    psf=,
    name="fermi-lat"
)

In [ ]:
fermi_dataset

## Perform joint fit
Now that we have our two dataset, we can try to fit them together.

First, we need to put them together in a `Datasets` object.

In [ ]:
datasets = 

In [ ]:
datasets

Then we also need two background model specific to Fermi:

In [ ]:
diffuse_iso = create_fermi_isotropic_diffuse_model(
    filename='iso_P8R3_SOURCE_V3_v1.txt'
)
diffuse_iso.datasets_names = fermi_dataset.name
print(diffuse_iso)

In [ ]:
template_diffuse = TemplateSpatialModel.read(
    filename="gll_iem_v07.fits",
    normalize=False
)
diffuse_iem = SkyModel(
    spectral_model=PowerLawNormSpectralModel(),
    spatial_model=template_diffuse,
    name='diffuse-iem'
)

diffuse_iem.datasets_names = fermi_dataset.name

And the other sources in the region of interest. For this, first read the 4FGL Fermi catalog via `CATALOG_REGISTRY.get_cls("4fgl")()`


In [ ]:
catalog_4fgl = 

Select only the sources inside the dataset spatial geometry:

In [ ]:
in_geom = geom.to_image().contains(catalog_4fgl.positions)
catalog_4fgl_gc = catalog_4fgl[in_geom]
models_4fgl_gc = catalog_4fgl_gc.to_models()


But these are too many to fit all together. Freeze all parameters of all sources except for bright and closeby sources (this will need to be optimized for your different analysis cases in real applications)

In [ ]:
separations  = catalog_4fgl_gc.positions.separation(target_pos).deg
significance = catalog_4fgl_gc.table['Signif_Avg']

for k,s in enumerate(models_4fgl_gc):
    #set norm and spectral par free for close sources and very bright ones:
    if separations[k]<5 and significance[k]>30:
        free_pars = ['norm','amplitude','alpha','index','index_1','index_2','beta','lambda_']
    #set norm free for close, but not bright sources
    elif separations[k]<5:
        free_pars = ['norm','amplitude']
    #do not free anything for all other sources
    else:
        free_pars = []
    #Apply the freeze/non-freeze settings
    for j,p in enumerate(models_4fgl_gc[k].parameters):
        if p.name in free_pars:
            p.frozen = False
        else:
            p.frozen = True
    

## Make the final model

Combine all Fermi relevant models (the 4FGL sources and the two diffuse models) into one `Models` object

In [ ]:
model_list = Models(list(models_4fgl_gc)+[diffuse_iem]+[diffuse_iso])

Our source of interest is registered under its 4FGL name

In [ ]:
source_name_4fgl = "4FGL J2158.8-3013"

Assign the models to our datasets. The Fermi-LAT dataset needs all models, the H.E.S.S. dataset only our source.

In [ ]:
fermi_dataset.models = model_list
hess_dataset.models = [models_4fgl_gc[source_name_4fgl]]

## Fitting the data:

Try to fit the data using an `ExpCutoffPowerLawSpectralModel` including EBL absorption for our source model. The start values are already given.

In [ ]:
spectral_model = XXX(
    index=2.0,
    amplitude=1e-12 * u.Unit("cm-2 s-1 TeV-1"),
    reference=1 * u.TeV,
    lambda_=0.1 / u.TeV,
    alpha=1,
)

In [ ]:
abs_model = EBLAbsorptionNormSpectralModel.read_builtin()
abs_model.redshift.value = 0.116


In [ ]:
spectral_model_abs = 
datasets.models[source_name_4fgl].spectral_model = spectral_model_abs

In [ ]:
fit_joint = 
result_joint = 

In [ ]:
result_joint

In [ ]:
spectral_model

Plot the fitted spectral model with its error band using `plot_error`

In [ ]:
spectral_model.plot_error(
    energy_bounds=,
    sed_type="e2dnde",
    label="EXP_Cutoff_PL",
)
plt.legend()

Produce the flux-points for each dataset using `FluxPointsEstimator`. For Fermi-LAT, you can merge every 2 energy bins (`edges[::2]`) to save time.

In [ ]:
fp_hess = FluxPointsEstimator(
    energy_edges=,
    source=source_name_4fgl,
    selection_optional="all"
).run(datasets=)

In [ ]:
#Merging every 2 energy bins to save time for the demo here
fp_fermi = FluxPointsEstimator(
    energy_edges=,
    source=source_name_4fgl,
    selection_optional="all"
).run(datasets=)

Plot the model (with EBL absorption) together with both sets of flux points. Note: `FluxPoints.plot()` changes the x-axis range, so set the axis limits at the end.

In [ ]:
spectral_model_abs.plot_error(
    energy_bounds=[0.1 * u.GeV, 10 * u.TeV],
    sed_type="e2dnde",
    label="EXP_Cutoff_PL",
)
# Add the flux points of H.E.S.S. and Fermi-LAT


plt.ylim(1e-13, 5e-9)
plt.xlim(1e-4, 100)

## Open problem:
Fit a physical model instead of Exp Cuoff PL. E.g. Inverse Compton using the Naima Plugin inside Gammapy (careful, for real cases beaming needs to be taken into account for blazars and externals plugings such as jetset or agnpy are more recommendable)

### 1. Define the physical model
In blazars, the high-energy peak is usually explained by synchrotron self-Compton (SSC) emission: the electrons in the jet emit synchrotron photons in the magnetic field, and the same electrons up-scatter these photons to gamma-ray energies (Inverse Compton).

We model this with Naima: an exponential cutoff power-law electron spectrum inside a spherical region with magnetic field `B` and radius `radius`. `NaimaSpectralModel` computes the synchrotron photon density inside the region and uses it as seed photons for the Inverse Compton emission.

This is a simplified model: it ignores the Doppler boosting of the jet, so the fitted parameters are not physical. For real blazar modelling use dedicated packages such as agnpy or jetset.

In [ ]:
import naima
from astropy.cosmology import Planck18
from gammapy.modeling.models import NaimaSpectralModel



### 2. Joint fit of the H.E.S.S. and Fermi-LAT data
As for the exponential cutoff power law, replace the spectral model of our source, this time by the SSC model times the EBL absorption, and fit both datasets together.

Hints:
- The SSC model is much slower to evaluate than an analytic model. Think about which parameters you really need to fit.


### 3. Plot the result
Compare the intrinsic model (as emitted by the source) with the absorbed one (as seen on Earth).